# Meta-Ensemble v2: Predicting Win/Loss (not hand category)

The previous meta-ensemble notebook stacked RF + ANN + Monte Carlo to predict **hand category** (NOTHING/PAIR/etc.) and found stacking added nothing over the ANN alone. The reasoning there: Monte Carlo estimates **win probability vs. a random opponent**, which is a genuinely different question than "what category will this hand become" — so it didn't transfer well as a feature for that task.

This notebook flips it around: **use RF's and ANN's predicted hand-category probabilities as features to predict the Monte Carlo win probability** — i.e., build a fast (~microseconds) surrogate for win-equity estimation that doesn't require running thousands of Monte Carlo simulations at inference time. This is the task Monte Carlo was actually built for, so it's a much more natural fit.

**Data note:** this uses a freshly-generated set of 2,000 randomly-dealt hands (not the original CSV) — fully random 7-card deals from a standard deck, split into Preflop/Flop/Turn stage-rows the same way as before. This keeps the evaluation completely independent of both the training data and any dependency on re-uploading files.

In [ ]:
import numpy as np, pandas as pd, random, time, json
from collections import Counter
from itertools import combinations
import joblib

## 1. Recap: hand evaluator + Monte Carlo (from the previous notebook)

Same correctness-checked evaluator and equity simulator as before — see the previous notebook (`poker_meta_ensemble.ipynb`) for the sanity checks.

In [ ]:
def hand_rank_5(cards):
    ranks = sorted([r for s, r in cards], reverse=True)
    suits = [s for s, r in cards]
    rank_counts = Counter(ranks)
    counts = sorted(rank_counts.items(), key=lambda x: (-x[1], -x[0]))
    count_vals = [c for r, c in counts]
    is_flush = len(set(suits)) == 1
    unique_ranks = sorted(set(ranks), reverse=True)
    is_straight = False; straight_high = None
    if len(unique_ranks) == 5:
        if unique_ranks[0] - unique_ranks[4] == 4:
            is_straight = True; straight_high = unique_ranks[0]
        elif unique_ranks == [14, 5, 4, 3, 2]:
            is_straight = True; straight_high = 5
    if is_straight and is_flush:
        return (8, straight_high)
    elif count_vals[0] == 4:
        return (7, counts[0][0], counts[1][0])
    elif count_vals[0] == 3 and count_vals[1] == 2:
        return (6, counts[0][0], counts[1][0])
    elif is_flush:
        return (5,) + tuple(sorted(ranks, reverse=True))
    elif is_straight:
        return (4, straight_high)
    elif count_vals[0] == 3:
        kickers = sorted([r for r, c in counts if c == 1], reverse=True)
        return (3, counts[0][0]) + tuple(kickers)
    elif count_vals[0] == 2 and count_vals[1] == 2:
        pairs = sorted([r for r, c in counts if c == 2], reverse=True)
        kicker = [r for r, c in counts if c == 1]
        return (2,) + tuple(pairs) + tuple(kicker)
    elif count_vals[0] == 2:
        kickers = sorted([r for r, c in counts if c == 1], reverse=True)
        return (1, counts[0][0]) + tuple(kickers)
    else:
        return (0,) + tuple(sorted(ranks, reverse=True))

def best_of_seven(cards7):
    best = None
    for combo in combinations(cards7, 5):
        r = hand_rank_5(list(combo))
        if best is None or r > best:
            best = r
    return best

full_deck = [(s, r) for s in range(4) for r in range(2, 15)]

def estimate_equity(hero_hole, board, num_sims, rgen):
    known = set(hero_hole + board)
    deck = [cd for cd in full_deck if cd not in known]
    cards_needed = 5 - len(board)
    wins = ties = losses = 0
    for _ in range(num_sims):
        rgen.shuffle(deck)
        opp = deck[0:2]
        sim_board = board + deck[2:2 + cards_needed]
        hr = best_of_seven(hero_hole + sim_board)
        orr = best_of_seven(opp + sim_board)
        if hr > orr: wins += 1
        elif orr > hr: losses += 1
        else: ties += 1
    return wins / num_sims, ties / num_sims, losses / num_sims

## 2. Generate held-out hands with true, exact targets

For each randomly-dealt hand, deal a full 7-card outcome (2 hole + 5 board) up front, then reveal it progressively to build the three stage-rows — matching the original dataset's semantics exactly:

| Stage | Input features (what RF/ANN see) | True target label |
|---|---|---|
| Preflop | 2 hole cards | best hand from hole + flop (5 cards) |
| Flop | hole + flop (5 cards) | best hand from hole + turn (6 cards) |
| Turn | hole + flop + turn (6 cards) | best hand from hole + river (7 cards) |

Monte Carlo equity is computed using only the cards known at the **input** stage (matching what RF/ANN see), so it's a fair basis for comparison.

In [ ]:
FEAT_NAMES_NOTE = None  # (feature engineering function is identical to the main notebook)

def longest_run(rs):
    if not rs: return 0
    rs = sorted(set(rs)); best = 1; cur = 1
    for i in range(1, len(rs)):
        if rs[i] == rs[i-1] + 1: cur += 1; best = max(best, cur)
        else: cur = 1
    return best

def build_features(hole, community):
    all_cards = hole + community
    padded = all_cards + [(0, 0)] * (7 - len(all_cards))
    flat = {}
    for i, (suit, rank) in enumerate(padded, start=1):
        flat[f'c{i}_rank'] = rank; flat[f'c{i}_suit'] = suit
    ranks = [r for s, r in all_cards]; suits = [s for s, r in all_cards]
    rc = Counter(ranks); sc = Counter(suits)
    flat['max_rank_count'] = max(rc.values()) if rc else 0
    flat['num_pairs'] = sum(1 for c in rc.values() if c == 2)
    flat['has_trips'] = int(any(c >= 3 for c in rc.values()))
    flat['has_quads'] = int(any(c >= 4 for c in rc.values()))
    flat['max_suit_count'] = max(sc.values()) if sc else 0
    flat['distinct_ranks'] = len(rc); flat['distinct_suits'] = len(sc)
    flat['high_card'] = max(ranks) if ranks else 0
    flat['low_card'] = min(ranks) if ranks else 0
    flat['rank_sum'] = min(sum(ranks), 127)
    flat['rank_span'] = (max(ranks) - min(ranks)) if ranks else 0
    flat['rank_std'] = float(np.std(ranks)) if len(ranks) > 1 else 0.0
    run1 = longest_run(ranks); alt = [1 if r == 14 else r for r in ranks]
    flat['straight_potential'] = max(run1, longest_run(alt))
    h_ranks = [hole[0][1], hole[1][1]]; h_suits = [hole[0][0], hole[1][0]]
    flat['is_pocket_pair'] = int(h_ranks[0] == h_ranks[1])
    flat['is_suited_hole'] = int(h_suits[0] == h_suits[1])
    flat['hole_gap'] = abs(h_ranks[0] - h_ranks[1])
    flat['hole_high'] = max(h_ranks)
    flat['num_cards_seen'] = len(all_cards)
    return flat

def category_label(cards):
    best = best_of_seven(cards) if len(cards) > 5 else hand_rank_5(cards)
    cat = best[0]
    return 9 if (cat == 8 and best[1] == 14) else cat   # split out royal flush as its own class

N_HANDS = 2000
NUM_SIMS = 200
rng = random.Random(7)
rng_mc = random.Random(42)

rows, mc_win_l, mc_tie_l, mc_lose_l = [], [], [], []
for n in range(N_HANDS):
    deal = rng.sample(full_deck, 7)
    hole = deal[0:2]; flop = deal[2:5]; turn4 = flop + deal[5:6]; river5 = flop + deal[5:6] + deal[6:7]
    stage_specs = [(hole, [], flop, 0), (hole, flop, turn4, 1), (hole, turn4, river5, 2)]
    for hole_c, input_comm, target_comm, stage_code in stage_specs:
        feat = build_features(hole_c, input_comm)
        feat['stage'] = stage_code
        feat['target'] = category_label(hole_c + target_comm)
        rows.append(feat)
        w, t, l = estimate_equity(hole_c, input_comm, NUM_SIMS, rng_mc)
        mc_win_l.append(w); mc_tie_l.append(t); mc_lose_l.append(l)

meta_rows = pd.DataFrame(rows)
meta_rows['mc_win'] = mc_win_l; meta_rows['mc_tie'] = mc_tie_l; meta_rows['mc_lose'] = mc_lose_l
print("meta_rows:", meta_rows.shape)

meta_rows: (6000, 37)


## 3. Get RF + ANN predictions on these hands (sanity-check they still perform as expected)

In [ ]:
from tensorflow.keras.models import load_model

X_TRAIN_COLUMNS = ['c1_rank', 'c2_rank', 'c3_rank', 'c4_rank', 'c5_rank', 'c6_rank', 'c7_rank',
    'rank_std', 'max_rank_count', 'num_pairs', 'has_trips', 'has_quads', 'max_suit_count',
    'distinct_ranks', 'distinct_suits', 'high_card', 'low_card', 'rank_sum', 'rank_span',
    'straight_potential', 'is_pocket_pair', 'is_suited_hole', 'hole_gap', 'hole_high',
    'num_cards_seen', 'c1_suit_0', 'c1_suit_1', 'c1_suit_2', 'c1_suit_3', 'c2_suit_0',
    'c2_suit_1', 'c2_suit_2', 'c2_suit_3', 'c3_suit_0', 'c3_suit_1', 'c3_suit_2', 'c3_suit_3',
    'c4_suit_0', 'c4_suit_1', 'c4_suit_2', 'c4_suit_3', 'c5_suit_0', 'c5_suit_1', 'c5_suit_2',
    'c5_suit_3', 'c6_suit_0', 'c6_suit_1', 'c6_suit_2', 'c6_suit_3', 'c7_suit_0',
    'stage_0', 'stage_1', 'stage_2']

suit_cols = [f'c{i}_suit' for i in range(1, 8)]
base_cols = [col for col in meta_rows.columns if col not in ['mc_win','mc_tie','mc_lose','target']]
base_oh = pd.get_dummies(meta_rows[base_cols], columns=suit_cols+['stage'], prefix=suit_cols+['stage'])
base_oh = base_oh.reindex(columns=X_TRAIN_COLUMNS, fill_value=0)

best_rf = joblib.load("random_forest_model.joblib")
rf_probs = best_rf.predict_proba(base_oh)

scaler = joblib.load("scaler.joblib")
ann = load_model("ann_model.h5")
ann_probs = ann.predict(scaler.transform(base_oh.astype(np.float32)), verbose=0, batch_size=512)

label_encoder = joblib.load("label_encoder.joblib")
y_true = label_encoder.transform(meta_rows['target'])
print("RF alone accuracy on fresh random hands:", (rf_probs.argmax(axis=1) == y_true).mean())
print("ANN alone accuracy on fresh random hands:", (ann_probs.argmax(axis=1) == y_true).mean())

RF alone accuracy on fresh random hands: 0.6303333333333333
ANN alone accuracy on fresh random hands: 0.6335


Consistent with the ~63-64% seen on the original data — confirms the models generalize to genuinely new hands, not just memorized quirks of the training sample.

## 4. Assemble meta-features and retarget: predict win probability

**Meta-features (20 total):** RF's 10 class-probabilities + ANN's 10 class-probabilities.
**Meta-target:** `mc_win` — Monte Carlo's estimated win probability vs. a random opponent (continuous, 0 to 1). Also derive a binary `is_favorite = mc_win > 0.5` for the classification framing.

The goal: can RF+ANN's category beliefs alone (no simulation) predict what 200 Monte Carlo trials would say, at a fraction of the compute cost?

In [ ]:
rf_cols = [f'rf_p{cl}' for cl in label_encoder.classes_]
ann_cols = [f'ann_p{cl}' for cl in label_encoder.classes_]
meta_X = pd.DataFrame(rf_probs, columns=rf_cols)
for i, colname in enumerate(ann_cols):
    meta_X[colname] = ann_probs[:, i]

y_win = meta_rows['mc_win'].to_numpy()
y_favorite = (y_win > 0.5).astype(int)

from sklearn.model_selection import train_test_split
Xtr, Xte, ywtr, ywte, yftr, yfte = train_test_split(
    meta_X, y_win, y_favorite, test_size=0.3, random_state=42, stratify=y_favorite
)
print("meta-train:", Xtr.shape, "meta-test:", Xte.shape)
print("favorite class balance (train):", np.bincount(yftr) / len(yftr))

meta-train: (4200, 20) meta-test: (1800, 20)
favorite class balance (train): [0.56809524 0.43190476]


## 5. Meta-model 1: Linear Regression — predict continuous win probability

This is now the *right* tool for the job (unlike in the category-prediction notebook): `mc_win` is a genuine continuous quantity in [0,1], so ordinary least squares is a natural fit. Compared against a naive baseline that always predicts the training-set average win probability, to confirm the model is adding real signal.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

linreg = LinearRegression()
linreg.fit(Xtr, ywtr)
pred = np.clip(linreg.predict(Xte), 0, 1)
mae = mean_absolute_error(ywte, pred)
rmse = mean_squared_error(ywte, pred) ** 0.5
r2 = r2_score(ywte, pred)
print(f"Linear Regression: MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")

baseline_pred = np.full_like(ywte, ywtr.mean())
baseline_mae = mean_absolute_error(ywte, baseline_pred)
print(f"Baseline (always predict mean={ywtr.mean():.3f}): MAE={baseline_mae:.4f}")

Linear Regression: MAE=0.1264  RMSE=0.1605  R²=0.3583
Baseline (always predict mean=0.481): MAE=0.1604


**~21% lower error than the baseline**, and R²=0.36 means the model explains over a third of the variance in win probability — real, meaningful signal, unlike the category-prediction task.

## 6. Meta-model 2: Logistic Regression — predict favorite vs. underdog

Binary classification: will this hand end up with >50% win probability (`is_favorite`)? This is the natural use case for logistic regression.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

logreg = LogisticRegression(max_iter=2000)
logreg.fit(Xtr, yftr)
logreg_acc = accuracy_score(yfte, logreg.predict(Xte))
logreg_auc = roc_auc_score(yfte, logreg.predict_proba(Xte)[:, 1])
print(f"Logistic Regression: accuracy={logreg_acc:.4f}  AUC={logreg_auc:.4f}")

Logistic Regression: accuracy=0.7283  AUC=0.7971


**72.8% accuracy, 0.80 AUC** — well above the ~57% you'd get by always guessing the majority class, and clearly discriminative.

## 7. Meta-model 3: ANN — nonlinear regression

A small feedforward net with a sigmoid output (since win probability is bounded in [0,1]), trained to minimize MSE.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

ann_wl = models.Sequential([
    layers.Input(shape=(Xtr.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
ann_wl.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = ann_wl.fit(Xtr.to_numpy(), ywtr, validation_split=0.15, epochs=80, batch_size=64,
                      callbacks=[early_stop], verbose=2)

pred = ann_wl.predict(Xte.to_numpy(), verbose=0).flatten()
mae = mean_absolute_error(ywte, pred); rmse = mean_squared_error(ywte, pred)**0.5; r2 = r2_score(ywte, pred)
print(f"ANN: MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")

ann_favorite_pred = (pred > 0.5).astype(int)
ann_acc = accuracy_score(yfte, ann_favorite_pred)
ann_auc = roc_auc_score(yfte, pred)
print(f"ANN as favorite/underdog classifier: accuracy={ann_acc:.4f}  AUC={ann_auc:.4f}")

Epoch 1/80
56/56 - 2s - 28ms/step - loss: 0.0283 - mae: 0.1291 - val_loss: 0.0382 - val_mae: 0.1577
Epoch 5/80
56/56 - 0s - 5ms/step - loss: 0.0236 - mae: 0.1189 - val_loss: 0.0344 - val_mae: 0.1503
Epoch 9/80
56/56 - 0s - 4ms/step - loss: 0.0229 - mae: 0.1168 - val_loss: 0.0266 - val_mae: 0.1299
Epoch 12/80
56/56 - 0s - 5ms/step - loss: 0.0229 - mae: 0.1171 - val_loss: 0.0216 - val_mae: 0.1165
Epoch 16/80
56/56 - 0s - 6ms/step - loss: 0.0226 - mae: 0.1170 - val_loss: 0.0210 - val_mae: 0.1131
Epoch 20/80
56/56 - 0s - 5ms/step - loss: 0.0224 - mae: 0.1159 - val_loss: 0.0211 - val_mae: 0.1136
Epoch 24/80
56/56 - 0s - 4ms/step - loss: 0.0222 - mae: 0.1160 - val_loss: 0.0210 - val_mae: 0.1135
Epoch 26/80
56/56 - 0s - 6ms/step - loss: 0.0222 - mae: 0.1154 - val_loss: 0.0226 - val_mae: 0.1192
(early stopping restored best weights from epoch 16)
ANN: MAE=0.1149  RMSE=0.1471  R²=0.4610
ANN as favorite/underdog classifier: accuracy=0.7206  AUC=0.8190


## 8. Final comparison

In [ ]:
print(f"{'Model':30s}{'Task':22s}{'Metric':>12s}")
print(f"{'Baseline (mean)':30s}{'regression':22s}{'MAE=0.1604':>12s}")
print(f"{'Linear Regression':30s}{'regression':22s}{'MAE=0.1264':>12s}")
print(f"{'ANN':30s}{'regression':22s}{'MAE=0.1149':>12s}")
print()
print(f"{'Logistic Regression':30s}{'favorite y/n':22s}{'acc=72.8%':>12s}")
print(f"{'ANN (thresholded)':30s}{'favorite y/n':22s}{'acc=72.1%':>12s}")

Model                         Task                        Metric
Baseline (mean)                regression              MAE=0.1604
Linear Regression              regression              MAE=0.1264
ANN                            regression              MAE=0.1149

Logistic Regression           favorite y/n               acc=72.8%
ANN (thresholded)             favorite y/n               acc=72.1%


## 9. Conclusion

**This retargeting worked.** Unlike hand-category stacking (which added nothing over the ANN alone), predicting Monte Carlo's win probability from RF+ANN's category beliefs shows real, meaningful signal:

- **ANN regression** cuts prediction error by **28% vs. baseline** (MAE 0.115 vs 0.160) and explains **46% of the variance** (R²=0.46) in win probability — using only the two models' category-probability outputs, no simulation required.
- **Logistic Regression** correctly calls favorite-vs-underdog **72.8%** of the time (AUC 0.80), well above the ~57% majority-class floor.
- The **ANN edges out Linear Regression** on the regression task (0.115 vs 0.126 MAE), suggesting a mildly nonlinear relationship between hand-category beliefs and win equity — sensible, since e.g. "NOTHING with a flush draw" and "NOTHING with no outs" look identical to a pure category classifier but have very different win probabilities.

**Why this makes sense where the other direction didn't:** RF and ANN's category probabilities *are* informative about equity — knowing a hand is likely to become TWO PAIR tells you a lot about how it'll fare against a random opponent. But knowing the win probability doesn't tell you nearly as much about the specific category (a 60% favorite could be a strong PAIR or a modest STRAIGHT). Information flows better in this direction.

**Practical use:** this gives you a way to get an approximate equity estimate in the time it takes to run two already-trained models, instead of running hundreds of Monte Carlo trials — useful if equity needs to be estimated many times per second (e.g. live at a table across many simultaneous hands).

## 10. Save the win/loss meta-models

In [ ]:
joblib.dump(linreg, 'winloss_linear_regression.joblib')
joblib.dump(logreg, 'winloss_logistic_regression.joblib')
ann_wl.save('winloss_ann.h5')
print("Saved: winloss_linear_regression.joblib, winloss_logistic_regression.joblib, winloss_ann.h5")

Saved: winloss_linear_regression.joblib, winloss_logistic_regression.joblib, winloss_ann.h5
